Общие импорты

In [1]:
import datetime
import math
from copy import deepcopy

Задание 1

In [2]:
class Operation:
    _types_operation = {
        "deposit": "deposit",
        "withdraw": "withdraw",
        "interest": "interest",
    }
    _statuses_operation = {
        "success": "success",
        "fail": "fail",
    }

    @staticmethod
    def get_types_operation() -> dict[str, str]:
        return deepcopy(Operation._types_operation)

    @staticmethod
    def get_statuses_operation() -> dict[str, str]:
        return deepcopy(Operation._statuses_operation)

    def __init__(self, account_number: str, type_operation: str, statuses_operation: str, amount: float, current_balance: float) -> None:
        if not type_operation in Operation._types_operation:
            raise ValueError(f"Type operation must be either: {Operation._types_operation}")
        if not statuses_operation in Operation._statuses_operation:
            raise ValueError(f"Status operation must be either: {Operation._statuses_operation}")

        self._account_number: str = account_number
        self._type_operation: str = Operation._types_operation[type_operation]
        self._statuses_operation: str = Operation._statuses_operation[statuses_operation]
        self._amount: float = amount
        self._current_balance: float = current_balance
        self._date: datetime.datetime = datetime.datetime.now()

    @property
    def amount(self) -> float:
        return self._amount

    @property
    def date(self) -> datetime.datetime:
        return self._date

    def __str__(self) -> str:
        return (f"{self._account_number} "
                f"{self._type_operation} "
                f"{self._statuses_operation} "
                f"{self._amount} "
                f"{self._current_balance} "
                f"{self._date:%Y-%m-%d %H:%M:%S}")

    def __repr__(self) -> str:
        return f"{self.__str__()}"


class Account:
    _account_counter: int = 1000

    def __init__(self, account_holder: str, balance: float = 0) -> None:
        if not self.validate_account_holder(account_holder):
            raise ValueError(f"The account holder\'s name must be in the format \"First Name Last Name\".")
        if not math.isfinite(balance) or balance < 0:
            raise ValueError("Balance must be finite and non-negative")
        self.holder: str = account_holder
        self.account_number: str = f"ACC-{self._account_counter}"
        self._balance: float = balance
        self.operations_history: list[Operation] = []

        Account._account_counter += 1

    def log(self, type_operation: str, status: str, amount: float, current_balance: float) -> None:
        new_operation = Operation(
            self.account_number,
            type_operation,
            status,
            amount,
            current_balance,
        )
        self.operations_history.append(new_operation)

    def deposit(self, amount: float) -> None:
        if not math.isfinite(amount) or amount <= 0:
            raise ValueError("Amount must be finite and positive")

        new_balance: float = self._balance + amount
        if not math.isfinite(new_balance):
            raise ValueError("Resulting balance must be finite")

        self._balance = new_balance
        self.log("deposit", "success", amount, self._balance)

    def withdraw(self, amount: float) -> None:
        if not math.isfinite(amount) or amount <= 0:
            raise ValueError("Amount must be finite and positive")

        if amount > self._balance:
            self.log("withdraw", "fail", amount, self._balance)
            raise ValueError("Insufficient funds")
        else:
            self._balance -= amount
            self.log("withdraw", "success", amount, self._balance)

    def get_transaction_analysis(self, limit: int, min_amount: float = 0) -> list[Operation]:
        if limit < 0:
            raise ValueError("Limit must be non-negative")
        if not math.isfinite(min_amount) or min_amount < 0:
            raise ValueError("Minimum amount must be finite and non-negative")

        operations: list[Operation] = sorted(
            (operation for operation in reversed(self.get_history()) if operation.amount >= min_amount),
            key=lambda operation: operation.date,
            reverse=True,
        )
        return operations[:limit]

    @staticmethod
    def validate_account_holder(account_holder: str) -> bool:
        upper: str = "АБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯABCDEFGHIJKLMNOPQRSTUVWXYZ"
        letters: str = upper + upper.lower()
        parts: list[str] = account_holder.split(" ")

        return len(parts) == 2 and all(
            part.istitle() and all(char in letters for char in part)
            for part in parts)

    def get_balance(self) -> float:
        return self._balance

    def get_history(self) -> list[Operation]:
        return deepcopy(self.operations_history)

    def __str__(self) -> str:
        return f"{self.holder} {self.account_number} {self.get_balance()}"

    def __repr__(self) -> str:
        return f"{self.__str__()}"

Задание 2

In [3]:
class CheckingAccount(Account):
    account_type: str = "checking"


class SavingsAccount(Account):
    account_type: str = "savings"
    balance_threshold: int = 50

    def withdraw(self, amount: float) -> None:
        limit: float = self._balance * (self.balance_threshold / 100)
        if limit < amount <= self._balance:
            self.log("withdraw", "fail", amount, self._balance)
            raise ValueError(f"You cannot withdraw more than {self.balance_threshold}% of the balance.")

        super().withdraw(amount)

    def apply_interest(self, rate: float) -> None:
        if not math.isfinite(rate) or rate < 0:
            raise ValueError("Rate must be finite and non-negative")

        interest_amount: float = self._balance * (rate / 100)
        new_balance: float = self._balance + interest_amount
        if not math.isfinite(new_balance):
            raise ValueError("Resulting balance must be finite")

        self.log("interest", "success", interest_amount, new_balance)
        self._balance = new_balance